# Imports and helpers

In [36]:
from pathlib import Path
import json
import re
from typing import Any

import pandas as pd
from datasets import load_dataset
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

NOTEBOOK_DIR = Path.cwd()

# If notebook is in multilingual/notebooks, this gives repo root.
REPO_ROOT = NOTEBOOK_DIR.parents[1]
MULTILINGUAL_DIR = REPO_ROOT / "multilingual"

PROCESSED_ROOT = MULTILINGUAL_DIR / "data" / "processed" / "mera"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:    ", NOTEBOOK_DIR)
print("REPO_ROOT:       ", REPO_ROOT)
print("MULTILINGUAL_DIR:", MULTILINGUAL_DIR)
print("PROCESSED_ROOT:  ", PROCESSED_ROOT)

NOTEBOOK_DIR:     /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/notebooks
REPO_ROOT:        /scratch/workspace/emma/standard-project-m2-mystery-machine
MULTILINGUAL_DIR: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual
PROCESSED_ROOT:   /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera


In [37]:
HF_DATASET = "MERA-evaluation/MERA"

SELECTED_TASKS = [
    "mamuramu",
    "mathlogicqa",
    "ruopenbookqa",
    "ruworldtree",
    "parus",
]

DROPPED_TASKS = [
    "use", # independent MCQ, many right answers
    "rcb",
    "rwsd",
]

# we do not use test/public_test for now.
ALLOWED_SPLITS = {"train", "validation", "dev"}

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
CORRUPTION_RE = re.compile(r"[\ufffd\x00-\x08\x0b\x0c\x0e-\x1f]")

In [38]:
def has_corruption(x: Any) -> bool:
    return bool(CORRUPTION_RE.search(str(x)))


def clean_text(x: Any) -> str:
    if x is None:
        return ""
    return str(x).strip()


def to_json_str(x: Any) -> str:
    return json.dumps(x, ensure_ascii=False)


def safe_get(d: Any, key: str, default: Any = "") -> Any:
    if isinstance(d, dict):
        return d.get(key, default)
    return default


def format_abcd(question: str, options: dict[str, Any]) -> str:
    return (
        f"{question}\n\n"
        f"A) {clean_text(options.get('A'))}\n"
        f"B) {clean_text(options.get('B'))}\n"
        f"C) {clean_text(options.get('C'))}\n"
        f"D) {clean_text(options.get('D'))}"
    )


def make_record(
    *,
    subset: str,
    split: str,
    source_id: str,
    task_type: str,
    prompt: str,
    answer: str,
    metadata: dict[str, Any],
) -> dict[str, Any]:
    return {
        "dataset": "mera",
        "subset": subset.lower(),
        "language": "russian",
        "split": split,
        "source_id": source_id,
        "task_type": task_type,
        "prompt": clean_text(prompt),
        "answer": clean_text(answer),
        "metadata": metadata,
    }


def is_valid_record(r: dict[str, Any]) -> bool:
    fields = [r["prompt"], r["answer"]]
    metadata_str = to_json_str(r.get("metadata", {}))

    return (
        r["split"] in {"train", "val"}
        and r["prompt"] != ""
        and r["answer"] != ""
        and not any(has_corruption(x) for x in fields)
        and not has_corruption(metadata_str)
    )

## Task-specific helpers for the subsets

In [39]:
def format_mamuramu(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    question = clean_text(inp.get("text"))
    options = {
        "A": inp.get("option_a"),
        "B": inp.get("option_b"),
        "C": inp.get("option_c"),
        "D": inp.get("option_d"),
    }

    return make_record(
        subset="MaMuRAMu",
        split=split,
        source_id=f"MaMuRAMu_{split}_{idx}",
        task_type="mcq",
        prompt=format_abcd(question, options),
        answer=clean_text(ex["outputs"]).upper(),
        metadata={
            "subject": clean_text(inp.get("subject")),
            "domain": clean_text(meta.get("domain")),
            "id": meta.get("id", idx),
        },
    )


def format_mathlogicqa(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    question = clean_text(inp.get("text"))
    options = {
        "A": inp.get("option_a"),
        "B": inp.get("option_b"),
        "C": inp.get("option_c"),
        "D": inp.get("option_d"),
    }

    return make_record(
        subset="MathLogicQA",
        split=split,
        source_id=f"MathLogicQA_{split}_{idx}",
        task_type="mcq",
        prompt=format_abcd(question, options),
        answer=clean_text(ex["outputs"]).upper(),
        metadata={
            "task": clean_text(meta.get("task")),
            "id": meta.get("id", idx),
        },
    )


def format_ruopenbookqa(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    question = clean_text(inp.get("text") or inp.get("question"))
    options = {
        "A": inp.get("option_a"),
        "B": inp.get("option_b"),
        "C": inp.get("option_c"),
        "D": inp.get("option_d"),
    }

    return make_record(
        subset="ruOpenBookQA",
        split=split,
        source_id=f"ruOpenBookQA_{split}_{idx}",
        task_type="mcq",
        prompt=format_abcd(question, options),
        answer=clean_text(ex["outputs"]).upper(),
        metadata={
            "id": meta.get("id", idx),
        },
    )


def format_ruworldtree(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    question = clean_text(inp.get("question") or inp.get("text"))
    options = {
        "A": inp.get("option_a"),
        "B": inp.get("option_b"),
        "C": inp.get("option_c"),
        "D": inp.get("option_d"),
    }

    return make_record(
        subset="ruWorldTree",
        split=split,
        source_id=f"ruWorldTree_{split}_{idx}",
        task_type="mcq",
        prompt=format_abcd(question, options),
        answer=clean_text(ex["outputs"]).upper(),
        metadata={
            "id": meta.get("id", idx),
            "exam_name": clean_text(meta.get("exam_name")),
            "school_grade": meta.get("school_grade", None),
            "knowledge_type": clean_text(meta.get("knowledge_type")),
        },
    )


def format_parus(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    q_type = clean_text(meta.get("task"))
    if q_type == "cause":
        q_line = "Что является причиной описанной ситуации?"
    elif q_type == "effect":
        q_line = "Что является следствием описанной ситуации?"
    else:
        q_line = "Какой вариант наиболее правдоподобен?"

    raw_answer = clean_text(ex["outputs"])
    answer_map = {"1": "A", "2": "B"}
    if raw_answer not in answer_map:
        raise ValueError(f"Unexpected PARus answer: {raw_answer!r}")

    prompt = (
        f"{clean_text(inp.get('premise'))}\n\n"
        f"{q_line}\n\n"
        f"A) {clean_text(inp.get('choice1'))}\n"
        f"B) {clean_text(inp.get('choice2'))}"
    )

    return make_record(
        subset="parus",
        split=split,
        source_id=f"parus_{split}_{idx}",
        task_type="mcq",
        prompt=prompt,
        answer=answer_map[raw_answer],
        metadata={
            "task": q_type,
            "id": meta.get("id", idx),
            "raw_answer": raw_answer,
            "option_a": clean_text(inp.get("choice1")),
            "option_b": clean_text(inp.get("choice2")),
        },
    )

def format_rcb(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    raw_answer = clean_text(ex["outputs"])
    answer_map = {"1": "A", "2": "B", "3": "C"}
    if raw_answer not in answer_map:
        raise ValueError(f"Unexpected RCB answer: {raw_answer!r}")

    prompt = (
        f"Ситуация:\n{clean_text(inp.get('premise'))}\n\n"
        f"Гипотеза:\n{clean_text(inp.get('hypothesis'))}\n\n"
        "Выберите отношение гипотезы к ситуации:\n"
        "A) гипотеза следует из ситуации\n"
        "B) гипотеза противоречит ситуации\n"
        "C) гипотеза независима от ситуации"
    )

    return make_record(
        subset="rcb",
        split=split,
        source_id=f"rcb_{split}_{idx}",
        task_type="mcq",
        prompt=prompt,
        answer=answer_map[raw_answer],
        metadata={
            "genre": clean_text(meta.get("genre")),
            "verb": clean_text(meta.get("verb")),
            "negation": clean_text(meta.get("negation")),
            "id": meta.get("id", idx),
            "raw_answer": raw_answer,
            "option_a": "гипотеза следует из ситуации",
            "option_b": "гипотеза противоречит ситуации",
            "option_c": "гипотеза независима от ситуации",
        },
    )


# NB: rwsd is binary choice (Yes/No answers), we reformat this to MCQ (2 options), Yes --> A since Y/N pretty balanced, roughly 50/50
def format_rwsd(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    raw_answer = clean_text(ex["outputs"]).strip().lower()
    if raw_answer not in {"да", "нет"}:
        raise ValueError(f"Unexpected RWSD answer: {ex['outputs']!r}")

    answer = "A" if raw_answer == "да" else "B"

    prompt = (
        f"Текст:\n{clean_text(inp.get('text'))}\n\n"
        f"Фрагмент 1: {clean_text(inp.get('span1_text'))}\n"
        f"Фрагмент 2: {clean_text(inp.get('span2_text'))}\n\n"
        "Относится ли фрагмент 2 к фрагменту 1?\n\n"
        "A) Да\n"
        "B) Нет"
    )

    return make_record(
        subset="rwsd",
        split=split,
        source_id=f"rwsd_{split}_{idx}",
        task_type="mcq",
        prompt=prompt,
        answer=answer,
        metadata={
            "id": meta.get("id", idx),
            "span1_index": inp.get("span1_index", None),
            "span2_index": inp.get("span2_index", None),
            "raw_answer": clean_text(ex["outputs"]),
            "option_a": "Да",
            "option_b": "Нет",
        },
    )



def format_use(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    task = clean_text(inp.get("task"))
    text = clean_text(inp.get("text"))
    choices = clean_text(inp.get("choices"))
    additional_text = clean_text(inp.get("additional_text"))
    use_type = clean_text(meta.get("type"))

    parts = []
    if text:
        parts.append(f"Текст:\n{text}")
    if task:
        parts.append(f"Задание:\n{task}")
    if additional_text:
        parts.append(f"Дополнительный текст:\n{additional_text}")
    if choices:
        parts.append(f"Варианты ответа:\n{choices}")

    prompt = "\n\n".join(parts)
    if use_type.startswith("multiple_choice") or use_type == "matching":
        task_type = "numeric"
    else:
        task_type = "short_answer"

    return make_record(
        subset="use",
        split=split,
        source_id=f"use_{split}_{idx}",
        task_type=task_type,
        prompt=prompt,
        answer=clean_text(ex["outputs"]),
        metadata={
            "id": meta.get("id", idx),
            "id_task": clean_text(meta.get("id_task")),
            "variant": meta.get("variant", None),
            "score": meta.get("score", None),
            "type": use_type,
        },
    )

## Formatter registry

In [40]:
FORMATTERS = {
    "use": format_use,
    "mamuramu": format_mamuramu,
    "mathlogicqa": format_mathlogicqa,
    "ruopenbookqa": format_ruopenbookqa,
    "ruworldtree": format_ruworldtree,
    "parus": format_parus,
    "rcb": format_rcb,
    "rwsd": format_rwsd,
}

In [41]:
for task in SELECTED_TASKS:
    print(f"\n=== {task} ===")
    ds = load_dataset(HF_DATASET, task)
    print(ds)
    for split_name in ds.keys():
        print(f"  {split_name}: {len(ds[split_name])}")


=== mamuramu ===
DatasetDict({
    train: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 285
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 4248
    })
})
  train: 285
  test: 4248

=== mathlogicqa ===
DatasetDict({
    train: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 680
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 1143
    })
})
  train: 680
  test: 1143

=== ruopenbookqa ===
DatasetDict({
    train: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 2338
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 400
    })
})
  train: 2338
  test: 400

=== ruworldtree ===
DatasetDict({
    train: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
      

# Normalized dataframe

In [42]:
# build normalized mera dataframe
rows = []
rejected_rows = []

for task in SELECTED_TASKS:
    print(f"\nProcessing {task}...")
    ds = load_dataset(HF_DATASET, task)
    formatter = FORMATTERS[task]

    for raw_split in ds.keys():
        if raw_split not in ALLOWED_SPLITS:
            print(f"  skipping split {raw_split}")
            continue

        out_split = "val" if raw_split in {"validation", "dev"} else "train"

        for idx, ex in enumerate(ds[raw_split]):
            try:
                record = formatter(ex, out_split, idx)

                if is_valid_record(record):
                    rows.append(record)
                else:
                    rejected_rows.append({
                        "subset": task,
                        "split": out_split,
                        "source_id": record.get("source_id", f"{task}_{out_split}_{idx}"),
                        "reason": "invalid_record",
                        "raw": to_json_str(ex),
                        "record": to_json_str(record),
                    })

            except Exception as e:
                rejected_rows.append({
                    "subset": task,
                    "split": out_split,
                    "source_id": f"{task}_{out_split}_{idx}",
                    "reason": repr(e),
                    "raw": to_json_str(ex),
                    "record": "",
                })

clean_raw = pd.DataFrame(rows)
rejected = pd.DataFrame(rejected_rows)

print("Raw clean rows:", len(clean_raw))
print("Rejected rows: ", len(rejected))

display(clean_raw.head())


Processing mamuramu...
  skipping split test

Processing mathlogicqa...
  skipping split test

Processing ruopenbookqa...
  skipping split test

Processing ruworldtree...
  skipping split test

Processing parus...
  skipping split test
Raw clean rows: 3918
Rejected rows:  0


,dataset,subset,language,split,source_id,task_type,prompt,answer,metadata
0,mera,mamuramu,russian,train,MaMuRAMu_train_0,mcq,"Какое число больше остальных: 73; 52,5; -5; 75; 32,83?\n\nA) 73\nB) 52,5\nC) -5\nD) 75",D,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 0}"
1,mera,mamuramu,russian,train,MaMuRAMu_train_1,mcq,"Сколько целых чисел больше 3, но меньше 6?\n\nA) 0\nB) 1\nC) 2\nD) Бесконечно много",C,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 1}"
2,mera,mamuramu,russian,train,MaMuRAMu_train_2,mcq,Как называется результат при сложении?\n\nA) Делимое\nB) Произведение\nC) Значение суммы\nD) Разность,C,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 2}"
3,mera,mamuramu,russian,train,MaMuRAMu_train_3,mcq,В часе 60 минут. Чему равна четверть часа?\n\nA) 30 минут\nB) 25 минут\nC) 20 минут\nD) 15 минут,D,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 3}"
4,mera,mamuramu,russian,train,MaMuRAMu_train_4,mcq,"Как называется часть прямой, ограниченная двумя точками?\n\nA) Луч\nB) Линия\nC) Отрезок\nD) Прямая",C,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 4}"


In [43]:
# filter to mcq-like and numeric tasks only
KEEP_TASK_TYPES = {"mcq", "numeric", "binary_choice"}

clean = clean_raw[clean_raw["task_type"].isin(KEEP_TASK_TYPES)].copy()
excluded = clean_raw[~clean_raw["task_type"].isin(KEEP_TASK_TYPES)].copy()

print("Raw clean rows:", len(clean_raw))
print("Kept rows:     ", len(clean))
print("Excluded rows: ", len(excluded))

print("\nKept task types:")
display(
    clean.groupby(["subset", "task_type"])
    .size()
    .reset_index(name="n")
    .sort_values(["subset", "task_type"])
)

print("\nExcluded task types:")
if len(excluded):
    display(
        excluded.groupby(["subset", "task_type"])
        .size()
        .reset_index(name="n")
        .sort_values(["subset", "task_type"])
    )
else:
    print("No excluded rows.")

Raw clean rows: 3918
Kept rows:      3918
Excluded rows:  0

Kept task types:


,subset,task_type,n
0,mamuramu,mcq,285
1,mathlogicqa,mcq,680
2,parus,mcq,500
3,ruopenbookqa,mcq,2338
4,ruworldtree,mcq,115



Excluded task types:
No excluded rows.


In [44]:
clean.head()

,dataset,subset,language,split,source_id,task_type,prompt,answer,metadata
0,mera,mamuramu,russian,train,MaMuRAMu_train_0,mcq,"Какое число больше остальных: 73; 52,5; -5; 75; 32,83?\n\nA) 73\nB) 52,5\nC) -5\nD) 75",D,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 0}"
1,mera,mamuramu,russian,train,MaMuRAMu_train_1,mcq,"Сколько целых чисел больше 3, но меньше 6?\n\nA) 0\nB) 1\nC) 2\nD) Бесконечно много",C,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 1}"
2,mera,mamuramu,russian,train,MaMuRAMu_train_2,mcq,Как называется результат при сложении?\n\nA) Делимое\nB) Произведение\nC) Значение суммы\nD) Разность,C,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 2}"
3,mera,mamuramu,russian,train,MaMuRAMu_train_3,mcq,В часе 60 минут. Чему равна четверть часа?\n\nA) 30 минут\nB) 25 минут\nC) 20 минут\nD) 15 минут,D,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 3}"
4,mera,mamuramu,russian,train,MaMuRAMu_train_4,mcq,"Как называется часть прямой, ограниченная двумя точками?\n\nA) Луч\nB) Линия\nC) Отрезок\nD) Прямая",C,"{'subject': 'Математика', 'domain': 'elementary_mathematics', 'id': 4}"


In [45]:
# inspect final counts
print("Counts by subset and split:")
display(
    clean.groupby(["subset", "split"])
    .size()
    .reset_index(name="n")
    .sort_values(["subset", "split"])
)

print("\nTotal by subset:")
display(
    clean.groupby("subset")
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

print("\nRejected rows:")
if len(rejected):
    display(
        rejected.groupby(["subset", "reason"])
        .size()
        .reset_index(name="n")
        .sort_values(["subset", "n"], ascending=[True, False])
    )
else:
    print("No rejected rows.")

Counts by subset and split:


,subset,split,n
0,mamuramu,train,285
1,mathlogicqa,train,680
2,parus,train,400
3,parus,val,100
4,ruopenbookqa,train,2338
5,ruworldtree,train,115



Total by subset:


,subset,n
3,ruopenbookqa,2338
1,mathlogicqa,680
2,parus,500
0,mamuramu,285
4,ruworldtree,115



Rejected rows:
No rejected rows.


In [46]:
# inspect a few final examples
def show_examples(df: pd.DataFrame, subset: str, n: int = 2):
    sub = df[df["subset"].str.lower() == subset.lower()]
    if sub.empty:
        print(f"No rows for {subset}")
        return

    sample = sub.sample(min(n, len(sub)), random_state=42)
    for _, row in sample.iterrows():
        print("=" * 80)
        print(f"subset: {row['subset']} | split: {row['split']} | task_type: {row['task_type']}")
        print("\nPROMPT:")
        print(row["prompt"])
        print("\nANSWER:")
        print(row["answer"])
        print("\nMETADATA:")
        print(row["metadata"])
        print()

for task in SELECTED_TASKS:
    if task in set(clean["subset"]):
        print(f"\n\n### {task}")
        show_examples(clean, task, n=2)



### mamuramu
subset: mamuramu | split: train | task_type: mcq

PROMPT:
Как называется отношение длины окружности к диаметру?

A) Число π
B) Число e
C) Число φ
D) Число γ

ANSWER:
A

METADATA:
{'subject': 'Математика', 'domain': 'high_school_mathematics', 'id': 9}

subset: mamuramu | split: train | task_type: mcq

PROMPT:
Clostridium tetani вызывает следующий тип инфекции

A) бактериемию
B) вирусемию
C) токсинемию
D) септицемию

ANSWER:
C

METADATA:
{'subject': 'Медицина', 'domain': 'virology', 'id': 249}



### mathlogicqa
subset: mathlogicqa | split: train | task_type: mcq

PROMPT:
Если 1 равен d плюс 2. 2, умноженное на d, плюс 2 равно a. То следует вычислить значение a.

A) 0
B) 1
C) -1
D) 4

ANSWER:
A

METADATA:
{'task': 'math', 'id': 647}

subset: mathlogicqa | split: train | task_type: mcq

PROMPT:
Семнадцать рабочих производят на 46 продуктов больше, чем 15 рабочих. Сколько продуктов произведут 20 рабочих?

A) 500
B) 440
C) 450
D) 460

ANSWER:
D

METADATA:
{'task': 'logic', 'i

In [47]:
# inspect binary-choice label distributions
binary = clean_raw[clean_raw["task_type"] == "binary_choice"].copy()

display(
    binary.groupby(["subset", "answer"])
    .size()
    .reset_index(name="n")
    .sort_values(["subset", "answer"])
)

display(
    binary.groupby(["subset", "split", "answer"])
    .size()
    .reset_index(name="n")
    .sort_values(["subset", "split", "answer"])
)

rwsd = clean_raw[clean_raw["subset"] == "rwsd"].copy()

display(
    rwsd.groupby(["split", "answer"])
    .size()
    .reset_index(name="n")
    .sort_values(["split", "answer"])
)

,subset,answer,n


,subset,split,answer,n


,split,answer,n


In [48]:
# check which hf splits were used
for task in SELECTED_TASKS:
    ds = load_dataset(HF_DATASET, task)
    allowed = [s for s in ds.keys() if s in ALLOWED_SPLITS]
    skipped = [s for s in ds.keys() if s not in ALLOWED_SPLITS]
    print(f"{task:15} allowed={allowed} skipped={skipped}")

mamuramu        allowed=['train'] skipped=['test']
mathlogicqa     allowed=['train'] skipped=['test']
ruopenbookqa    allowed=['train'] skipped=['test']
ruworldtree     allowed=['train'] skipped=['test']
parus           allowed=['train', 'validation'] skipped=['test']


In [52]:
for subset in ["parus", "rcb", "rwsd", "use"]:
    sub = clean[clean["subset"] == subset]
    print(f"\n{subset}")
    display(sub["answer"].value_counts().sort_index())
    display(sub[["prompt", "answer", "metadata"]].sample(min(3, len(sub)), random_state=42))


parus


answer
A    250
B    250
Name: count, dtype: int64

,prompt,answer,metadata
3779,Мужчина зацепился рукавом за перекладину забора.\n\nЧто является следствием описанной ситуации?\n\nA) Он лез через забор.\nB) Рукав порвался.,B,"{'task': 'effect', 'id': 361, 'raw_answer': '2', 'option_a': 'Он лез через забор.', 'option_b': 'Рукав порвался.'}"
3491,Его одежда сидит слишком свободно.\n\nЧто является причиной описанной ситуации?\n\nA) Он купил её на распродаже.\nB) Он сбросил вес.,B,"{'task': 'cause', 'id': 73, 'raw_answer': '2', 'option_a': 'Он купил её на распродаже.', 'option_b': 'Он сбросил вес.'}"
3792,У человека наболело.\n\nЧто является причиной описанной ситуации?\n\nA) Его жена критикует его.\nB) Его мать скончалась.,B,"{'task': 'cause', 'id': 374, 'raw_answer': '2', 'option_a': 'Его жена критикует его.', 'option_b': 'Его мать скончалась.'}"



rcb


Series([], Name: count, dtype: int64)

...

Series([], Name: count, dtype: int64)

,prompt,answer,metadata



use


Series([], Name: count, dtype: int64)

,prompt,answer,metadata


# Save

In [58]:
# save final processed files
clean_to_save = clean.copy()
excluded_to_save = excluded.copy()
rejected_to_save = rejected.copy()

clean_to_save["metadata"] = clean_to_save["metadata"].apply(to_json_str)
if len(excluded_to_save):
    excluded_to_save["metadata"] = excluded_to_save["metadata"].apply(to_json_str)

clean_path = PROCESSED_ROOT / "mera_clean_all.csv"
excluded_path = PROCESSED_ROOT / "mera_excluded_short_answer.csv"
rejected_path = PROCESSED_ROOT / "mera_rejected_all.csv"

clean_to_save.to_csv(clean_path, index=False, encoding="utf-8")
excluded_to_save.to_csv(excluded_path, index=False, encoding="utf-8")
rejected_to_save.to_csv(rejected_path, index=False, encoding="utf-8")


# save jsonl / json versions
jsonl_path = PROCESSED_ROOT / "mera_clean_all.jsonl"
json_path = PROCESSED_ROOT / "mera_clean_all.json"
clean_to_save.to_json(
    jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False,
)
clean_to_save.to_json(
    json_path,
    orient="records",
    indent=2,
    force_ascii=False,
)

print(f"Saved clean JSONL: {jsonl_path}")
print(f"Saved clean JSON:  {json_path}")

print(f"Saved clean:    {clean_path}")
print(f"Saved excluded: {excluded_path}")
print(f"Saved rejected: {rejected_path}")

for subset, sub in clean_to_save.groupby("subset"):
    csv_out_path = PROCESSED_ROOT / f"mera_clean_{subset}.csv"
    jsonl_out_path = PROCESSED_ROOT / f"mera_clean_{subset}.jsonl"
    json_out_path = PROCESSED_ROOT / f"mera_clean_{subset}.json"
    sub.to_csv(csv_out_path, index=False, encoding="utf-8")
    sub.to_json(jsonl_out_path, orient="records", lines=True, force_ascii=False)
    sub.to_json(json_out_path, orient="records", indent=2, force_ascii=False)
    print(f"Saved {subset}: {len(sub)} rows -> {out_path}")

Saved clean JSONL: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_all.jsonl
Saved clean JSON:  /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_all.json
Saved clean:    /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_all.csv
Saved excluded: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_excluded_short_answer.csv
Saved rejected: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_rejected_all.csv
Saved mamuramu: 285 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_ruworldtree.csv
Saved mathlogicqa: 680 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_ruworldtree.csv
Saved parus: 500 rows -> /scratch/wor

# Final check

In [55]:
df = pd.read_csv(PROCESSED_ROOT / "mera_clean_all.csv")

display(df.groupby(["subset", "split"]).size().reset_index(name="n"))
display(df.groupby(["subset", "task_type"]).size().reset_index(name="n"))
display(df["task_type"].value_counts())
print(df.columns.tolist())
print("has source_id:", "source_id" in df.columns)

,subset,split,n
0,mamuramu,train,285
1,mathlogicqa,train,680
2,parus,train,400
3,parus,val,100
4,ruopenbookqa,train,2338
5,ruworldtree,train,115


,subset,task_type,n
0,mamuramu,mcq,285
1,mathlogicqa,mcq,680
2,parus,mcq,500
3,ruopenbookqa,mcq,2338
4,ruworldtree,mcq,115


task_type
mcq    3918
Name: count, dtype: int64

['dataset', 'subset', 'language', 'split', 'source_id', 'task_type', 'prompt', 'answer', 'metadata']
has source_id: True


In [57]:
assert df.columns.tolist() == [
    "dataset",
    "subset",
    "language",
    "split",
    "source_id",
    "task_type",
    "prompt",
    "answer",
    "metadata",
]

assert df["prompt"].notna().all()
assert df["answer"].notna().all()
assert set(df["language"]) == {"russian"}
assert set(df["task_type"]) == {"mcq"}
assert set(df["answer"]).issubset({"A", "B", "C", "D"})
assert not df["prompt"].str.strip().eq("").any()
assert not df["answer"].str.strip().eq("").any()
assert not (df["task_type"] == "short_answer").any()
assert not (df["task_type"] == "numeric").any()
assert set(df["language"]) == {"russian"}
assert df["prompt"].notna().all()
assert df["answer"].notna().all()

In [59]:
# inspect random processed examples from selected subjects only
SAMPLE_N = 10
RANDOM_STATE = 42

SELECTED_SUBJECT_SUBSETS = [
    "mamuramu",
    "mathlogicqa",
    "ruopenbookqa",
    "ruworldtree",
    "parus",
]

sample_lines = []

for subset in SELECTED_SUBJECT_SUBSETS:
    sub = clean_raw[clean_raw["subset"].str.lower() == subset.lower()].copy()

    if sub.empty:
        print(f"No rows for {subset}")
        continue

    sample = sub.sample(n=min(SAMPLE_N, len(sub)), random_state=RANDOM_STATE)

    header = f"\n\n{'=' * 100}\n{subset.upper()} | sampled {len(sample)} / {len(sub)} examples\n{'=' * 100}"
    print(header)
    sample_lines.append(header)

    for i, (_, row) in enumerate(sample.iterrows(), start=1):
        block = (
            f"\n\n--- Example {i} ---\n"
            f"dataset:   {row['dataset']}\n"
            f"subset:    {row['subset']}\n"
            f"language:  {row['language']}\n"
            f"split:     {row['split']}\n"
            f"source_id: {row['source_id']}\n"
            f"task_type: {row['task_type']}\n"
            f"answer:    {row['answer']}\n"
            f"metadata:  {to_json_str(row['metadata'])}\n"
            f"\nPROMPT:\n{row['prompt']}\n"
        )

        print(block)
        sample_lines.append(block)

sample_txt = "\n".join(sample_lines)

sample_path = PROCESSED_ROOT / "mera_v7_random_samples_by_subset.txt"
sample_path.write_text(sample_txt, encoding="utf-8")

print(f"\nSaved random samples to: {sample_path}")



MAMURAMU | sampled 10 / 285 examples


--- Example 1 ---
dataset:   mera
subset:    mamuramu
language:  russian
split:     train
source_id: MaMuRAMu_train_9
task_type: mcq
answer:    A
metadata:  {"subject": "Математика", "domain": "high_school_mathematics", "id": 9}

PROMPT:
Как называется отношение длины окружности к диаметру?

A) Число π
B) Число e
C) Число φ
D) Число γ



--- Example 2 ---
dataset:   mera
subset:    mamuramu
language:  russian
split:     train
source_id: MaMuRAMu_train_249
task_type: mcq
answer:    C
metadata:  {"subject": "Медицина", "domain": "virology", "id": 249}

PROMPT:
Clostridium tetani вызывает следующий тип инфекции

A) бактериемию
B) вирусемию
C) токсинемию
D) септицемию



--- Example 3 ---
dataset:   mera
subset:    mamuramu
language:  russian
split:     train
source_id: MaMuRAMu_train_157
task_type: mcq
answer:    A
metadata:  {"subject": "Общество", "domain": "professional_psychology", "id": 157}

PROMPT:
Какую деятельность в наибольшей степени мож